# Подготовка датасета: Кот Бегемот

In [ ]:
!pip install -q openai

In [ ]:
import re, json
from openai import OpenAI

API_KEY = "sk-..."  # <-- вставь свой ключ OpenAI
CHARACTER = "Кот Бегемот"
ALIASES = ["Бегемот", "бегемот", "кот", "Кот", "котище"]

## 1. Загрузка и предобработка книги

In [ ]:
with open("/content/master_margarita.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

raw_text = raw_text.replace("\r", "\n")
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text)

print(f"Загружено {len(raw_text)} символов")

## 2. Парсинг реплик Бегемота

In [ ]:
VERBS = [
    "сказал", "сказала", "произнёс", "произнес", "ответил", "ответила",
    "спросил", "спросила", "проговорил", "заметил", "добавил",
    "воскликнул", "крикнул", "прошептал", "подтвердил", "возразил",
    "пробормотал", "перебил", "объяснил", "согласился", "отозвался",
    "осведомился", "поинтересовался", "отвечал", "приказал", "шепнул",
    "завопил", "закричал", "промяукал", "промурлыкал", "попросил",
]

lines = raw_text.split("\n")
speeches = []

for i, line in enumerate(lines):
    s = line.strip()
    if not s.startswith("—") and not s.startswith("–"):
        continue

    ctx = s
    if i + 1 < len(lines): ctx += " " + lines[i+1].strip()
    if i > 0: ctx = lines[i-1].strip() + " " + ctx

    has_verb = any(v in ctx.lower() for v in VERBS)
    is_char = any(a.lower() in ctx.lower() for a in ALIASES)

    speech = re.sub(r"^[—–]\s*", "", s)
    for v in VERBS:
        speech = re.split(rf"\s*[,.]?\s*[—–]\s*{v}\b.*$", speech, maxsplit=1)[0]
    speech = speech.strip()

    if len(speech) > 10:
        speeches.append((i, speech, is_char and has_verb))

In [ ]:
qa_book = []
seen = set()
generic_q = ["Что скажешь?", "И что же?", "Расскажи подробнее.",
             "Как так?", "Ты уверен?", "Что ты имеешь в виду?",
             "Продолжай.", "А дальше?"]

char_indices = [idx for idx, (_, _, is_c) in enumerate(speeches) if is_c]

for pos in char_indices:
    _, answer, _ = speeches[pos]
    if answer in seen:
        continue
    seen.add(answer)

    question = None
    for j in range(pos - 1, max(pos - 5, -1), -1):
        _, prev_text, prev_is_char = speeches[j]
        if not prev_is_char and len(prev_text) > 5:
            question = prev_text
            break

    qa_book.append({
        "question": question or generic_q[len(qa_book) % len(generic_q)],
        "answer": answer,
    })

print(f"Извлечено диалогов из книги: {len(qa_book)}")
for p in qa_book[:5]:
    print(f"  Q: {p['question'][:80]}")
    print(f"  A: {p['answer'][:100]}\n")

## 3. Аугментация через OpenAI

In [ ]:
client = OpenAI(api_key=API_KEY)
examples = "\n".join(f"- «{p['answer']}»" for p in qa_book[:15])

TOPICS = [
    ["еда и напитки", "хулиганство и озорство", "философия жизни"],
    ["люди и их глупость", "магия и сверхъестественное", "культура и искусство"],
    ["дружба и верность", "деньги и жадность", "трусость и храбрость"],
    ["современные технологии", "работа и карьера", "путешествия"],
    ["любовь и отношения", "правда и ложь", "советы на каждый день"],
    ["погода", "мода и внешний вид", "домашние животные"],
]

augmented = []

for i, topics in enumerate(TOPICS):
    topics_str = ", ".join(topics)
    print(f"Батч {i+1}/{len(TOPICS)}: {topics_str}")

    prompt = f"""Ты — эксперт по персонажу {CHARACTER} из «Мастера и Маргариты» Булгакова.

Примеры его реплик:
{examples}

Сгенерируй 10 пар «вопрос человека → ответ в стиле {CHARACTER}».
Темы: {topics_str}.
Ответы: 1-3 предложения, на русском, в характерной хамоватой и остроумной манере Бегемота.

Верни ТОЛЬКО JSON-массив:
[{{"question": "...", "answer": "..."}}]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        pairs = json.loads(raw)
        augmented.extend(pairs)
        print(f"  -> {len(pairs)} пар")
    except Exception as e:
        print(f"  ОШИБКА: {e}")

print(f"\nСгенерировано диалогов через LLM: {len(augmented)}")

## 4. Формат для SFTTrainer

In [ ]:
all_qa = qa_book + augmented

SYSTEM = (
    f"Ты — {CHARACTER} из романа «Мастер и Маргарита» Булгакова. "
    f"Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью."
)
IM_END = "<|im_end|>"

sft_dataset = []
for p in all_qa:
    q, a = p["question"].strip(), p["answer"].strip()
    if not q or not a:
        continue
    sft_dataset.append({
        "prompt": f"<|im_start|>system\n{SYSTEM}{IM_END}\n<|im_start|>user\n{q}{IM_END}\n<|im_start|>assistant\n",
        "completion": f"{a}{IM_END}",
    })

print(f"Из книги: {len(qa_book)}, сгенерировано: {len(augmented)}, итого: {len(sft_dataset)}")

## 5. Сохранение

In [ ]:
with open("dataset_behemot_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_qa, f, ensure_ascii=False, indent=2)

with open("dataset_behemot_sft.json", "w", encoding="utf-8") as f:
    json.dump(sft_dataset, f, ensure_ascii=False, indent=2)

print("Сохранено: dataset_behemot_raw.json, dataset_behemot_sft.json")

## 6. Загрузка в HF Dataset

In [ ]:
from datasets import Dataset

ds = Dataset.from_list(sft_dataset)
split = ds.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(split['train'])}, Val: {len(split['test'])}")
print(split["train"][0])